In [0]:
import requests
import pandas as pd

url = "https://api.economicdata.alberta.ca/api/data?code=3caa6978-9647-4eb3-8de8-34a3abc06427"

data = requests.get(url).json()

df = pd.DataFrame(data)

display(df)

Date,Type,Value
2007-01-01T00:00:00,Conventional Oil,2707743.4
2007-01-01T00:00:00,Non-Conventional Oil,5604533.2
2007-01-01T00:00:00,Total oil production,8312276.6
2007-02-01T00:00:00,Non-Conventional Oil,5414892.8
2007-02-01T00:00:00,Conventional Oil,2450594.2
2007-02-01T00:00:00,Total oil production,7865487.0
2007-03-01T00:00:00,Conventional Oil,2703836.8
2007-03-01T00:00:00,Non-Conventional Oil,6082110.5
2007-03-01T00:00:00,Total oil production,8785947.3
2007-04-01T00:00:00,Total oil production,8030452.2


In [0]:
df = df.rename(columns={
    "Date": "date",
    "Type": "oil_type",
    "Value": "production_volume"
})

df = df[df["oil_type"] != "Total oil production"]
df["date"] = pd.to_datetime(df["date"])

In [0]:
dim_oil_type = df[["oil_type"]].drop_duplicates().reset_index(drop=True)

dim_oil_type["type_id"] = dim_oil_type.index + 1

In [0]:
dim_date = pd.DataFrame()

dim_date["date"] = df["date"].unique()

dim_date["year"] = dim_date["date"].dt.year
dim_date["month"] = dim_date["date"].dt.month
dim_date["quarter"] = dim_date["date"].dt.quarter

In [0]:
fact_table = df.merge(dim_oil_type, on="oil_type")

fact_table = fact_table[[
    "date",
    "type_id",
    "production_volume"
]]

In [0]:
%pip install snowflake-connector-python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import snowflake.connector

conn = snowflake.connector.connect(
    user="USERNAME",
    password = "PASSWORD",
    role = "ROLE",
    account="ACCOUNTID",
    warehouse="COMPUTE_WH",
    database="ENERGY_ANALYTICS",
    schema="ENERGY_SCHEMA"
)

print("Connected to Snowflake!")

Connected to Snowflake!


In [0]:
dim_oil_type.columns = dim_oil_type.columns.str.upper()
dim_date.columns = dim_date.columns.str.upper()
fact_table.columns = fact_table.columns.str.upper()

dim_date["DATE"] = dim_date["DATE"].dt.date
fact_table["DATE"] = fact_table["DATE"].dt.date

In [0]:
from snowflake.connector.pandas_tools import write_pandas

write_pandas(conn, dim_oil_type, "DIM_OIL_TYPE")
write_pandas(conn, dim_date, "DIM_DATE")
write_pandas(conn, fact_table, "FACT_OIL_PRODUCTION")
